In [0]:

from pyspark.sql.functions import regexp_replace
from pyspark.sql.types import *
from pyspark.sql import functions as f
from pyspark.sql.window import Window


data = [
    (101, 'Rahul ', 'rahul@gmail.com', 'BANGALORE ', 50000, '98765-43210', '2025-01-02'),
    (101, 'Rahul ', 'rahul@gmail.com', 'BANGALORE', 50000, '98765-75210', '2025-01-01'),
    (101, 'Rahul ', 'rahul@gmail.com', 'BANGALORE', 50000, '98765-75210', '2025-01-01'),
    (102, 'Amit', None, 'Pune', 60000, '98765-55210', '2025-01-05'),
    (103, None, 'john@gmail.com', 'Chennai', 70000, '98765-00210', '2025-01-04'),
    (104, 'DAVID', 'david@gmail.com', None, None, '98765-00910', '2025-01-03'),
    (104, 'DAVID', 'david@gmail.com', None, None, '98765-88810', '2025-01-10'),
    (104, 'DAVID', 'david@gmail.com', None, None, '98765-88810', '2025-01-10'),
    (105, None, None, None, 80000, '99865-88810', '2025-01-14'),
    (None, 'Ravi', 'ravi@gmail.com', 'Hyderabad', 55000, '900615-88810', '2025-01-12')
]

schema = StructType([
    StructField('customer_id', IntegerType()),
    StructField('customer_name', StringType()),
    StructField('customer_email', StringType()),
    StructField('customer_city', StringType()),
    StructField('customer_salary', IntegerType()),
    StructField('customer_phone', StringType()),
    StructField('load_dt', StringType())
])

customers = spark.createDataFrame(data=data, schema=schema)

# Remove records which is having NUll in customer_id
customers = customers.dropna(subset='customer_id')
customers = customers.filter(f.col('customer_id').isNotNull())
# convert customers load_dt string to date
customers = customers.withColumn('load_dt', f.col('load_dt').cast(DateType()))
# Drop Duplicates
customers = customers.distinct()
# pick latest records and remove duplicates
w = Window.partitionBy('customer_id').orderBy(f.col('load_dt').desc())
customers = customers.withColumn('rn', f.row_number().over(w)).where(f.col('rn') == 1)
# Clean records fill Null with Unknown
customers = customers.fillna({
    'customer_name': 'Unknown',
    'customer_email': 'Unknown',
    'customer_city': 'Unknown',
    'customer_salary': 0,
}).select('customer_id', 'customer_name', 'customer_email', 'customer_city', 'customer_salary', 'customer_phone',
          'load_dt')

customers = customers.withColumn('customer_name', f.lower(f.trim('customer_name'))).withColumn('customer_city',
                                                                                               f.lower(f.trim(
                                                                                                   'customer_city')))

customers = customers.withColumn('customer_phone', regexp_replace('customer_phone', '-', ''))
customers.show()